In [ ]:
%pip -q install openai pandas numpy scipy scikit-learn tqdm

In [ ]:
from google.colab import drive, userdata
from IPython.display import display
from pathlib import Path
from datetime import datetime, timezone
import base64
import json
import os
import re
import time

import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from sklearn.metrics import mean_absolute_error
from tqdm.auto import tqdm
from openai import OpenAI

RUN_DEV_API_CALLS = True
RUN_TEST_API_CALLS = False
MODEL = 'gpt-5.5'
REASONING_EFFORT = 'none'
IMAGE_DETAIL = 'original'
MAX_RETRIES = 3

DEMONSTRATION_IDS = [
    'img_004_v3', 'img_023_v5', 'img_035_v4',
    'img_027_v4', 'img_010_v2', 'img_024_v2',
]

drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/Dr. Lulwah - Ahmed/ImageEVAl')
PROJECT_DIR = ROOT / 'ImageEval2026_Task2_CRAI_Bench'
DATA_DIR = PROJECT_DIR / 'data'
EXPERIMENT_ROOT = PROJECT_DIR / 'cea_direct_fewshot_v1'
CACHE_DIR = EXPERIMENT_ROOT / 'cache'
OUTPUT_DIR = EXPERIMENT_ROOT / 'outputs'
CACHE_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

client = OpenAI(api_key=userdata.get('openai'))

In [ ]:
def parse_base_id(instance_id):
    return re.sub(r'_v\d+$', '', str(instance_id))

def find_image(folder, stem):
    for suffix in ['.png', '.jpg', '.jpeg', '.webp']:
        path = folder / f'{stem}{suffix}'
        if path.exists():
            return str(path)

def load_split(split, gold=False):
    folder = DATA_DIR / split
    frame = pd.read_csv(folder / 'captions.tsv', sep='\t')
    if gold:
        labels = pd.read_csv(folder / 'gold_human.tsv', sep='\t')
        frame = frame.merge(labels[['id', 'CRAI_CEA']], on='id')
    frame['id'] = frame['id'].astype(str)
    frame['base_id'] = frame['id'].map(parse_base_id)
    frame['ref_image_path'] = frame['base_id'].map(
        lambda value: find_image(folder / 'imgs' / 'ref', value)
    )
    frame['generated_image_path'] = frame['id'].map(
        lambda value: find_image(folder / 'imgs' / 'generated', value)
    )
    return frame

train_df = load_split('train', gold=True)
dev_df = load_split('dev', gold=True)
test_df = load_split('test') if (DATA_DIR / 'test' / 'captions.tsv').exists() else pd.DataFrame()
demo_rows = train_df.set_index('id', drop=False).loc[DEMONSTRATION_IDS]

In [ ]:
PROMPT_VERSION = 'direct-fewshot-cea-v1'

DIRECT_CEA_PROMPT = r"""
You are evaluating Cultural Element Accuracy (CEA) for CRAI-Bench.

You receive:
1. an authentic Qatari reference image,
2. the current caption used to generate an image, and
3. the generated image.

CEA asks: Are the expected cultural elements present and correctly depicted?

Use the labelled examples to follow the human scoring scale. Predict one continuous
score from 0.0 to 1.0. Evaluate CEA only; do not score contextual coherence,
cultural specificity, cultural integrity, hallucination penalty, aesthetics, or
general image quality as separate factors.

Return only this JSON object:
{"CRAI_CEA": <number from 0.0 to 1.0>}
"""

In [ ]:
def image_to_data_url(path):
    path = Path(path)
    mime = {'.png': 'image/png', '.jpg': 'image/jpeg',
            '.jpeg': 'image/jpeg', '.webp': 'image/webp'}[path.suffix.lower()]
    return f'data:{mime};base64,' + base64.b64encode(path.read_bytes()).decode()

def user_content(row, labelled=False):
    prefix = 'LABELLED EXAMPLE' if labelled else 'INSTANCE TO SCORE'
    return [
        {'type': 'input_text', 'text': f"{prefix}\nID: {row['id']}\nCaption: {row['caption']}"},
        {'type': 'input_image', 'image_url': image_to_data_url(row['ref_image_path']),
         'detail': IMAGE_DETAIL},
        {'type': 'input_image', 'image_url': image_to_data_url(row['generated_image_path']),
         'detail': IMAGE_DETAIL},
    ]

def messages(row):
    result = []
    for _, example in demo_rows.iterrows():
        result.extend([
            {'role': 'user', 'content': user_content(example, labelled=True)},
            {'role': 'assistant', 'content': json.dumps({'CRAI_CEA': float(example['CRAI_CEA'])})},
        ])
    result.append({'role': 'user', 'content': user_content(row)})
    return result

def cache_path(split):
    return CACHE_DIR / f'{split}_direct_fewshot.jsonl'

def load_cache(path):
    if not path.exists():
        return {}
    with path.open() as handle:
        records = [json.loads(line) for line in handle if line.strip()]
    return {record['id']: record for record in records}

def score(row):
    for attempt in range(MAX_RETRIES):
        try:
            response = client.responses.create(
                model=MODEL,
                reasoning={'effort': REASONING_EFFORT},
                input=[{'role': 'developer', 'content': DIRECT_CEA_PROMPT}, *messages(row)],
            )
            text = response.output_text
            value = json.loads(text[text.find('{'):text.rfind('}') + 1])
            return {'id': row['id'], 'CRAI_CEA': float(value['CRAI_CEA'])}
        except Exception:
            if attempt == MAX_RETRIES - 1:
                raise
            time.sleep(2 ** (attempt + 1))

def run_split(frame, split, run_calls):
    path = cache_path(split)
    cache = load_cache(path)
    for _, row in tqdm(frame.iterrows(), total=len(frame), desc=split):
        if row['id'] not in cache and run_calls:
            cache[row['id']] = score(row)
            with path.open('a') as handle:
                handle.write(json.dumps(cache[row['id']]) + '\n')
    return pd.DataFrame([cache[x] for x in frame['id'] if x in cache])

In [ ]:
dev_predictions = run_split(dev_df, 'dev', RUN_DEV_API_CALLS)
if len(dev_predictions) == len(dev_df):
    evaluation = dev_df[['id', 'CRAI_CEA']].merge(
        dev_predictions, on='id', suffixes=('_gold', '_prediction')
    )
    metrics = pd.DataFrame([{
        'spearman': spearmanr(evaluation['CRAI_CEA_gold'], evaluation['CRAI_CEA_prediction']).statistic,
        'mae': mean_absolute_error(evaluation['CRAI_CEA_gold'], evaluation['CRAI_CEA_prediction']),
    }])
    display(metrics.round(4))
    dev_predictions.to_csv(OUTPUT_DIR / 'cea_direct_fewshot_dev_predictions.tsv', sep='\t', index=False)

if len(test_df):
    test_predictions = run_split(test_df, 'test', RUN_TEST_API_CALLS)
    if len(test_predictions) == len(test_df):
        test_predictions.to_csv(
            OUTPUT_DIR / 'cea_direct_fewshot_test_predictions.tsv', sep='\t', index=False
        )